<a href="https://colab.research.google.com/github/suwarnalatha-m/Task-2-Predictive-Analysis/blob/main/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
!pip install plotly
!pip install scikit-learn
import pandas as pd
import numpy as np

# Visualization (INTERACTIVE)
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Data loading with converted csv files
df = pd.read_csv("/content/online_retail.csv")

print(df.shape)
df.head()

In [ ]:
# Data Cleaning
# Remove missing customers
df = df.dropna(subset=['CustomerID'])

# Remove returns & invalid prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Create Revenue Column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [ ]:
import plotly.graph_objects as go

# Take readable sample
sample = df.sample(15).round(2)

fig = go.Figure(data=[go.Table(

    # Header styling
    header=dict(
        values=list(sample.columns),
        fill_color='#2C3E50',
        font=dict(color='white', size=14),
        align='center',
        height=35
    ),

    # Cell styling
    cells=dict(
        values=[sample[col] for col in sample.columns],
        fill_color=[['white','whitesmoke'] * 8],  # alternating rows
        align='left',
        font=dict(color='black', size=12),
        height=28
    )
)])

fig.update_layout(
    title="Online Retail Dataset Preview",
    width=1000,
    height=500
)

fig.show()

In [ ]:
# Convert Date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
# Create RFM Table
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
}).reset_index()

rfm.columns = ['CustomerID','Recency','Frequency','Monetary']

rfm.head()

In [ ]:
# Customer Spending Distribution
fig = px.histogram(
    rfm,
    x="Monetary",
    nbins=50,
    title="Customer Spending Distribution",
)
fig.show()

In [ ]:
# Recency vs Spending (Business Insight)
fig = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    size="Frequency",
    title="Customer Behavior Analysis",
    hover_data=['CustomerID']
)
fig.show()

In [ ]:
# Country Revenue Dashboard
country_sales = df.groupby('Country')['Revenue'].sum().reset_index()

fig = px.bar(
    country_sales.sort_values('Revenue', ascending=False),
    x='Country',
    y='Revenue',
    title="Revenue by Country"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Create Prediction Target

# Create Label
rfm['HighValueCustomer'] = np.where(
    rfm['Monetary'] > rfm['Monetary'].median(),
    1,
    0
)


In [ ]:
X = rfm[['Recency','Frequency','Monetary']]
y = rfm['HighValueCustomer']

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
# Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

# Model Evaluation
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(
    cm,
    text_auto=True,
    title="Confusion Matrix"
)
fig.show()

In [ ]:
# Feature Importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

fig = px.bar(
    importance,
    x='Feature',
    y='Importance',
    title="Feature Importance"
)
fig.show()

**Conclusion:**

A Random Forest classification model was developed to predict
high-value customers based on purchase behavior.

Feature engineering was performed using transaction data.
The model achieved good accuracy and demonstrated that
order quantity and number of purchases strongly influence
customer value prediction.